This is my fourth and final portfolio submission, it is based on ML4DA Week 9 Part 3 "Creating your own models".

First of all, the initial step.

Step 1. Setting things up, initialising, importing libraries and assigning aliases to them.
Step 2. Loading the data

I start by improting all the necessary dependencies and libraries before assigning some of them industry standard aliases (ensures ease of understanding for external reviewers and follows best practice).

I chose Matplotlib over Seaborn as I am certain it is compatible with this version of TensorFlow and irregardles Seaborn is built on top of Matplotlib (though the visualised data would look better if I chosen Seaborn, in the event it was compatibe with my tf and python version).

I also loaded in the data that the model will be trained on and used the head() method to print the first 5 rows of the CSV to ensure data hygeine (no corrupted or missing values).

Since there are new technologies being used I will describe them. Tensorflow is an open-source framework for ML using data flow graphs. The nodes within the graph are represntative of mathematical operations and the graph edges represent the multidimensional data arrays, called tensors, that flow between them. TF is heavily used by data scientists and gives the programmer control over parameters. It was developed by Google to help the development of ML models, mainly deep learning models by providing tools for building, training and cross platform deployment, it allows for easier model building as it has many levels of abrstraction (tf handles certain calculations and operations for the user). Keras is a high level API for building deep learning models and runs on top of tf (they are now shipped together), it is also good for quick development.

tf is more popular in industry, has keras built in, offers more features, and is more scalable so perhaps these were why it was chosen over Pytorch as it is a seperate libray without keras shipped with it.

Deep learning explanation:
DL is a subset of ML that is driven by multi-layered NNs (input layers, hidden layers, output layer) with design inspired by the structure of a human's brain (the original NN). DL is made up of algorithms that allow software to train itself to do tasks, like NLP and computer vision (image recognition). DL models power most cutting edge AI today like GenAI and self-driving cars.

In the code section Dense is a densely connected NN that inherits from layers (from keras.layers import Dense).

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from keras.layers import Dense
from keras.models import Sequential

# Loading the dataset
fuelcon_data = pd.read_csv("FuelConsumption.csv")

fuelcon_data.head()

,MODELYEAR,MAKE,MODEL,VEHICLECLASS,ENGINESIZE,CYLINDERS,TRANSMISSION,FUELTYPE,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
0,2014,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,2014,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,2014,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,2014,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,2014,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


Quick data sanity check to ensure no missing values. Sum up all the data entries that are empty (none are empty so all is well).

In [3]:
fuelcon_data.isnull().sum()

MODELYEAR                   0
MAKE                        0
MODEL                       0
VEHICLECLASS                0
ENGINESIZE                  0
CYLINDERS                   0
TRANSMISSION                0
FUELTYPE                    0
FUELCONSUMPTION_CITY        0
FUELCONSUMPTION_HWY         0
FUELCONSUMPTION_COMB        0
FUELCONSUMPTION_COMB_MPG    0
CO2EMISSIONS                0
dtype: int64

Step 3. Selecting Features and Target

Before beginning to build the model it is time to talk about neural network and advanced NN concepts, forward propagation is the core process in a neural network where input data passes through all the layers in a network from the input layer to the output layer and generates an output. The input data moves through every layer, input layer to hidden layer(s) to output layer, of the NN where each neuron adds bias, applies weighted sum and passes the result through an activation function and it also makes predictions. A neuron in this context is the basic unit that takes inputs, each neuron is governed by an activation function (sigmoid, tanh, softmax, etc.). Activation functions are necessary to remove linearity as forwardpropagation's "a=z" is a linear combination of inputs and weights as well as the bias, a is the output. The introduction of nonlinearity brought by the activation function is pivotal for enabling the network to learn more complex/intricate patterns. tanh gives better performance for multi-layer NNs than sigmoid but still suffered from the vanishing gradient problem before ReLU was introduced.

Since forwardpropagation is about predicting a single output/target or dependent variable based off of multiple features it is an example of regression (a supervised learning technique), forwardpropagation also shares the same equation as linear regression with y=mx+c or y=mx+b. M being the slope, y being the dependent variable, b being the intercept and x as the independent variable (y and x are also coordinates). It is supervised learning since there has to be labelled training data for it to predict an output/it needs to training data.

Backpropagation is about error/loss calculation, gradient calculation, and weight updates. In a supervised learning environment data is labnelled and also referred to as Ground Truyth (T), if y hat does not equal (!=) T then training is required. Backpropagation calculates the error E which is the difference between predicted (y hat) and actual values (T) and then uses the difference (the loss) to perform gradient descent on the biases and weights in the network. Backpropagation is a taining algorith that updates weights by minimising error.

With this explanation it is clear why I start with forwardpropagation as I have no loss if I have not built a predictor model, thus backpropagation cannot be done before forwardpropagation.
The specification stated that the model must predict vehicle CO2 emissions (target/dependent variable) from the Engine Size, Number of Cylinders and Combined Mileage (features), features/predictors/independent variables are the input variables used by the model for predictions, features are the individual measurable quantifiable characteristics or properties of a phenomenon being observed.

Only the specified fatures were selected from the csv and assigned to the "features" variable, any other ones would just add noise that would throw off the predictions. FUELCONSUMPTION_COMB (combined fuelcon for highways and cities) was chosen over FUELCONSUMPTION_COMB_MPG since mileage is the number of miles travelled during a car's lifetime whereas MPG is just how many miles you can go per gallon of fuel, what this means is that FUELCONSUMPTION_COMB gives us the total fuel used for the 2 types of roads the car was drivem on (highway and cities, at least the 2 types of roads that there are data for) and FUELCONSUMPTION_COMB_MPG just gives how many MPG. FUELCONSUMPTION_COMB_MPG does not say how many gallons of fuel were consumed in total as opposed to FUELCONSUMPTION_COMB saying how much fuel was consumed for a set distance on both road types. FUELCONSUMPTION_COMB is also directly proportional as fuel consumption goes up so do emissions, FUELCONSUMPTION_COMB_MPG seems less intuitive and less related.



In [4]:
# Predictors: Engine Size, Cylinders, Combined Mileage
features = fuelcon_data[['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_COMB']]
target = fuelcon_data['CO2EMISSIONS'] # the target the model will predict

features.head() # display first 5 rows and check that the correct columns were selected

,ENGINESIZE,CYLINDERS,FUELCONSUMPTION_COMB
0,2.0,4,8.5
1,2.4,4,9.6
2,1.5,4,5.9
3,3.5,6,11.1
4,3.5,6,10.6


Step 4. Normalisation of the data

I chose normalisation over scaling since I do not need to measure how far apart my data points are and I less so want to change the scale but more so want to ensure the data is normally distributed to minimise outliers.

I used Z-score normalisation, also known as standardisation since ML algorithms tend to struggle with datasets where features are on very different scales, if these features were not normalised the algorithm could potentialy give more importance to a feature with a larger scale, thus leading to inaccurate predictions. Mean and std calculations handled by pythonic abstractions.

I did not do a train test split here as week 8 said that was best practice for scaling (not sure if this was used inetrchangeably) and since the features selected are the only predictors there is no target leakage (like if it aleady knew CO2 emissions whilst trying to calculatet them), though an IBM states normalisation should be done on training data only the actual IBM Regression ipynb did not train test split so all should be fine (Classification did though). If this does lead to overfitting I will knwo it was due to not train/test splitting the data (80/20) before normalisation and will conclude it was bad practice.

In [5]:
# Neural networks perform significantly better when inputs are scaled
features_norm = (features - features.mean()) / features.std()
n_cols = features_norm.shape[1] # number of features/predictors

Step 5. Building a Neural Network and Compiling the model

Defined the function for the regression model. Sequential was used for the model since it is apt for a plain linear stack of layers. Not the best for multiple inputs allegedly.

"Dense(50, activation='relu')," means that there are 50 neurons with ReLU activation function, ReLU is used to remove linearity. Dense layers were used to ensure network interconnectivity where every neuron in the layer is connected to every neuron in the prior layer, this allows the deep learning models to learn complex patterns and relationships in data (ReLU allegedly makes networks sparse though). Softmax is not used in the output layer as this is not a classification model but a regression one. Sigmoid activation function is not used as it does not perform well with multi-layer NNs (as mentioned prior), and still suffers from vanishing gradient (not that much of a problem since I will not do backpropagation and instead use MAE to determine accuracy for predictions).

could also use leaky_relu instead of relu. Leaky ReLU handles ReLU limitations like dead neurons (if a neuron only gets negative inputs it outputs 0 and the gradient becomes 0 so it stops learning), faster learning than ReLU, and more. 

metrics='mae' added into the model.compile() method for a guage of accuracy of model predictions. input_shape used to say that I want the same amount of features as inputs for the input layer a.k.a I want a starting shape of 3, "model.add(Dense(1))" the 1 denotes one node/neuron as the output for the Output layer since linear regression predicts one output/dependent variable based off of multiple inputs/independent variables/X where X is a feature matrix.

input_shape allwos that variable to be automatically adjusted if I add more features later as opposed to manually specifying the amount of inputs I want.

optimizer='adam' ,eans automatically adjust the learning rate as training progresses (this optimiser is stochastic), loss='mean_squared_error' means square the loss actually (amendment to any erroneuos mention of using MAE as loss), mae (the average of the difference bewteen actual and predicted value) is used as a metric for the metrics class and just reports how far off the predictions are and serve as an accuracy metric/guide.


In [6]:
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Dense(50, activation='leaky_relu', input_shape=(n_cols,))) # input layer
    # hidden layers
    model.add(Dense(25, activation='leaky_relu'))
    model.add(Dense(25, activation='leaky_relu'))
    model.add(Dense(25, activation='leaky_relu'))
    model.add(Dense(25, activation='leaky_relu'))

    # output layer
    model.add(Dense(1))
    
    # compilng the model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics='mae')
    return model

Step 6. Training and testing the network

There is a split here as was shown in the Regression with Keras ipynb, meaning since some data is held as a testing/validation set here that there should not be an issue with my standardising earlier. fit() is used to train the model, it is not unique to scikit-learn.

An epoch is one full pass through the entire training dataset where every data sample (a single instance or observation within a dataset, it represents a unit of data collected for data anlysis or modeling) is passed through the model and its parameters are updated based on the calculated error. Since the current training data is 70% then the batch it is running on should be 70% of the original training dataset.

Message that the training is complete at the end to inform people. New line taken to make it more obvious.

In [ ]:
# build the model using the function call
model = regression_model()

# fit the model
model.fit(features_norm, target, validation_split=0.3, epochs=100, verbose=2) # assigned the value of this method to the history variable for future plot
print("\n-Training complete")


Epoch 1/100
24/24 - 1s - loss: 72649.5156 - mae: 261.8951 - val_loss: 62332.0078 - val_mae: 242.3593 - 733ms/epoch - 31ms/step
Epoch 2/100
24/24 - 0s - loss: 71927.8047 - mae: 260.7028 - val_loss: 61257.6211 - val_mae: 240.4420 - 59ms/epoch - 2ms/step
Epoch 3/100
24/24 - 0s - loss: 67804.0859 - mae: 253.8112 - val_loss: 55197.3828 - val_mae: 229.2809 - 61ms/epoch - 3ms/step
Epoch 4/100
24/24 - 0s - loss: 49074.9922 - mae: 215.9789 - val_loss: 31275.1621 - val_mae: 170.7784 - 60ms/epoch - 3ms/step
Epoch 5/100
24/24 - 0s - loss: 18073.0586 - mae: 118.6162 - val_loss: 11876.4727 - val_mae: 87.9343 - 60ms/epoch - 3ms/step
Epoch 6/100
24/24 - 0s - loss: 9065.2080 - mae: 76.4487 - val_loss: 8935.7979 - val_mae: 80.1666 - 60ms/epoch - 3ms/step
Epoch 7/100
24/24 - 0s - loss: 6584.2393 - mae: 66.7007 - val_loss: 6362.7314 - val_mae: 65.9369 - 59ms/epoch - 2ms/step
Epoch 8/100
24/24 - 0s - loss: 4735.3936 - mae: 56.1487 - val_loss: 4604.8013 - val_mae: 56.1372 - 61ms/epoch - 3ms/step
Epoch 9/100

Step 7. Experimentation

I will run expirements where I change at least these three things and then log my findings here:

1.  Increase or decreate number of neurons in hidden layers

Decreasing the amount of neurons in the hidden layer from 50 to 10 increases loss/error and MAE slightly
Increasing the amount of neurons in the hidden layer from 10 to 500 noticeably decreased loss and slightly decreased MAE on the first epoch, it then proceeded to get better but by the 100th epoch it became worse than my initial neuron value of 50 as val_loss rose by a lot and MAE increased.

2.  Add more hidden layers

I initially only had one but I will add 3 more hidden layers, each with 25 neurons (if they did not have the same amount of neurons one or some nodes would not be connected and would be sparser, meaning less understanding of complex patterns, this is not ideal since my dataset is not large enough for sparse NNs to be computationally better than dense ones), and document my finding.

I saw an initial increase in loss and MAE but then I saw a greater decrease as compared to only having one hidden layer, more layers mean more calculations before the final prediction, thus more accuracy. A lower MAE and loss means more accuracy too, if it is too low (perfect case of 0) then it is most likely overfitted. 

3.  Increase number of epochs

I increased the number of epochs to 300 from 100, this increased the run time from a few seconds to 20 seconds (rounded from 19.8) but the final epoch's loss was 280 from 417 anmd the mae was 8.22 from 30.53 thus it improved the model accuracy the most. Decreasing epochs would have the inverse effect. This shows that epochs do update the model as each iteration processes a batchm finds loss and then adjusts its weights and parameteers based off of that loss.


4. bonus experiment, Since I am not using backpropagation I will not permanently change the compile() method's optimiser to sgd as I am not using gradient descent to verify accurcay. I will experiment with it now to see what happens. All it did was corrput my values and make them "nan"


5.  Change activation functions

relu changed to leaky_relu

This initially brought positive changes but by the 300th epoch loss was down by 6 units, mae was up by around 0.7 and val_mae was up by around 0.13. Albeit val_loss was down by ~ 29 units.


Misc:
A verbose of 0 shows nothing, 1 shows an animated progress bar with dots (...) and === symbols, 2 just mentions the epochs.

Step 8. Plot val_mae and MAE

This part makes use of matplotlib for more data visualisation for visual feedback as this is important in industry since many people are visual learners, and there is a focus on building dashboards and other visual things.

The kernel keeps crashing and I could not fix it in time, I suspect this is due to a version conflict with tf and python 3, even when I tried to do the earliest week's extension installing tf through conda navigator did not work as I chose a modern Python version that tf did not support (not yet I assume). If it did work I would have shown a plot of the validation mae and the actual mae to see how accurate the model is and whether the lines overlap or superimpose on one another. Either that or the equivalent but with loss as opposed to mae.

The last reference in this notebook shows the error page I was taken to after clicking a hyperlink here about why my kernel crashed.

In [ ]:
# did not work as intended

: 

References:
https://www.geeksforgeeks.org/deep-learning/what-is-forward-propagation-in-neural-networks/
https://www.datacamp.com/tutorial/forward-propagation-neural-networks
https://www.geeksforgeeks.org/machine-learning/difference-between-back-propagation-and-feed-forward-neural-network/
https://www.geeksforgeeks.org/machine-learning/neural-networks-a-beginners-guide/
https://share.google/wF3hbvdb4n4CN65w6
https://www.geeksforgeeks.org/machine-learning/features-and-labels-in-supervised-learning-a-practical-approach/
https://www.geeksforgeeks.org/machine-learning/how-to-handle-noise-in-machine-learning/
https://www.nvidia.com/en-gb/glossary/tensorflow/
https://www.geeksforgeeks.org/python/introduction-to-tensorflow/
https://www.tensorflow.org/about
https://www.simplilearn.com/keras-vs-tensorflow-vs-pytorch-article
https://medium.com/@kanerika/keras-vs-pytorch-which-ml-framework-is-the-best-for-you-b3eb6451dd66
https://www.kaggle.com/code/alexisbcook/scaling-and-normalization
https://stats.stackexchange.com/questions/458579/should-i-normalize-all-data-prior-feeding-the-neural-network-models
https://www.geeksforgeeks.org/data-analysis/normalization-and-scaling/
https://www.geeksforgeeks.org/data-analysis/z-score-normalization-definition-and-examples/
https://h2o.ai/wiki/target-leakage/
https://www.ibm.com/think/topics/data-leakage-machine-learning
https://www.tensorflow.org/api_docs/python/tf/keras/utils/split_dataset
https://keras.io/guides/sequential_model/
https://www.geeksforgeeks.org/deep-learning/dense-layer-tf-keras-layers-dense-in-tensorflow/
https://www.geeksforgeeks.org/machine-learning/ml-classification-vs-regression/
https://www.geeksforgeeks.org/machine-learning/Leaky-Relu-Activation-Function-in-Deep-Learning/
https://www.geeksforgeeks.org/deep-learning/neural-network-node/
https://www.reddit.com/r/learnmachinelearning/comments/7uvdnr/x_y_is_feature_label_by_convention_in_machine/
https://keras.io/api/metrics/
https://keras.io/api/optimizers/adam/
https://www.ibm.com/think/topics/deep-learning
https://www.geeksforgeeks.org/machine-learning/epoch-in-machine-learning/
https://datascience.stackexchange.com/questions/56263/what-is-sample-and-feature
https://medium.com/@frankcool/data-samples-vs-features-in-machine-learning-b9d4b50c8fce
https://stackoverflow.com/questions/47902295/what-is-the-use-of-verbose-in-keras-while-validating-the-model
https://www.baeldung.com/cs/neural-networks-dense-sparse
https://en.wikipedia.org/wiki/NaN
https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
https://github.com/microsoft/vscode-jupyter/wiki/Kernel-crashes